In [1]:
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
import scipy as scp
import multiprocessing

/Users/andpetrouzeniou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def c_quantile(a, Sigma_Y, n_iter, rand):

    N = len(Sigma_Y)
    
    draws = rand.multivariate_normal(np.zeros(N), Sigma_Y, n_iter)
    diags = np.tile((1 / np.diagonal(Sigma_Y) ** (1/2)), (n_iter, 1))
    
    draws = abs(draws * diags)
    
    lst = np.amax(draws, axis=1)
    return np.quantile(np.array(lst), a)


In [3]:
def c_quantile_local(a, Sigma_Y, J, n_iter, rand):

    Sigma_Y = Sigma_Y[J,:]
    Sigma_Y = Sigma_Y[:,J]
    
    return c_quantile(a, Sigma_Y, n_iter, rand)


In [4]:
def d_quantile(b, k, Sigma_X, n_iter, rand):

    N = len(Sigma_X)
    
    draws = rand.multivariate_normal(np.zeros(N), Sigma_X, n_iter)
    lst = np.amax(draws, axis=1) - np.amax((-1 * draws), axis=1)

    return np.quantile(np.array(lst), b)


In [5]:
def r_l(y, x, top_k, Sigma_y, a):
    
    s = np.max((np.diagonal(Sigma_y)) ** 0.5)
    
    m = len(y)
    kth_largest = np.partition(arr, -top_k)[-top_k]
    
    delta_hat = []
    for j in range(len(y)):
        add = max([0, kth_largest - x[j]])
        delta_hat.append(min(add))
        
    delta_hat = np.array(delta_hat)
    delta_hat = np.flip(np.sort(delta_hat))
    
    alpha = a
    
    for j in range(len(y)):
        r_l_j = scp.stats.norm.ppf(1 - (alpha / (2 * (m - j)))) * s
        if delta_hat[j] <= 4 * r_l_j:
            return r_l_j
        else:
            alpha = alpha - 2 * (1 - scp.stats.norm.cdf((delta_hat[j] - r_l_j) / (3 * s)))
    

In [6]:
def r_u(y, x, top_k, Sigma_y, a):
    
    s = np.max((np.diagonal(Sigma_y)) ** 0.5)
    
    m = len(y)
    kth_largest = np.partition(arr, -top_k)[-top_k]
    
    delta_hat = []
    for j in range(len(y)):
        add = max([0, kth_largest - x[j]])
        delta_hat.append(min(add))
        
        
    delta_hat = np.array(delta_hat)
    delta_hat = np.flip(np.sort(delta_hat))
    
    alpha = a
    
    for j in range(len(y)):
        r_u_j = scp.stats.norm.ppf(1 - (alpha / (2 * (m - j)))) * s
        if delta_hat[j] <= 2 * r_u_j:
            return r_u_j
        else:
            S_inv_a = scp.stats.norm.ppf(1 - a / 2)
            alpha = alpha - 2 * (1 - scp.stats.norm.cdf((delta_hat[j] + S_inv_a) / (3 * s)))
    

In [7]:
def make_I_list(rank_list, X_temp, delta_L, delta_U):
    
    rank = np.max(np.array(rank_list))
    
    # need matrix A[i,j] = X_temp[j] + delta_L[j,i]
    
    X_temp_mat = np.tile(X_temp, (len(X_temp), 1))
    A = X_temp_mat + np.transpose(delta_L)
    
    np.fill_diagonal(A, np.nan)
    A = A[~np.isnan(A)].reshape(A.shape[0], A.shape[1] - 1)

    row_ranks = np.partition(A, -rank, axis=1)[:, -rank]
    
    indices = np.where(X_temp >= row_ranks, 1, 0)
        
    return indices


In [8]:
def p_j(delta, Sigma, k, top_k, n_iter, rand):
    
    Nk = len(Sigma)
    N = int(Nk / k)
    p_j_vec = [0 for j in range(N)]
    
    for i in range(n_iter):
        draw = rand.multivariate_normal(np.zeros(Nk), Sigma)
        
        for j in range(k):
            
            sub_draw = draw[(j * N):((j + 1) * N)]
            
            for r in range(N):
                if sub_draw[r] >= np.partition(np.delete(delta[j][:,r] + sub_draw, r), -(top_k-1))[-(top_k-1)]:
                    p_j_vec[r] += 1
        
    return np.array(p_j_vec) / n_iter
    

In [9]:
def J_hat(vec, lvl):
    
    tot = 0
    J_hat_lst = []
    
    not_used = [i for i in range(len(vec))]
    
    while tot < lvl:
        
        m = vec[not_used[0]]
        m_idx = not_used[0]
        for idx in not_used:
            if vec[idx] > m:
                m_idx = idx
                
        top = m_idx
    
        not_used.remove(top)
        J_hat_lst.append(top)
        
        if len(not_used) == 0:
            return J_hat_lst
        
        tot += vec[top]
        
    return np.array(J_hat_lst)
         

In [10]:
def c_2_quantile(ab, rank_list, Sigma, delta_L, delta_U, n_iter, rand):

    lst = []
    
    k = len(delta_L)
    p = int(len(Sigma) / (k + 1))
    
    Sigma_Y = Sigma[: p, : p]

    for a in range(n_iter):
        draw = rand.multivariate_normal(np.zeros(p * (k + 1)), Sigma)
        
        I_list = []
        
        for i in range(k):
                                        
            X_temp = draw.copy()
            X_temp = X_temp[((i + 1) * p) : ((i + 2) * p)]
            rank = np.max(np.array(rank_list))
            
            I_list.append(make_I_list([j+1 for j in range(rank_list[i])], X_temp, delta_L[i], delta_U[i]))

        s = []
        
        Y_temp = draw.copy()
        Y_temp = Y_temp[:p]
        
        all_indices = []
                                        
        for i in range(k):
                                        
            I_l = I_list[i]
            I_indices = np.where(I_l == 1)[0]
            all_indices = list(set(all_indices) | set(I_indices))
                
        for j in all_indices:
            if Sigma_Y[j,j] > 0:
                s.append(abs(Y_temp[j] / (Sigma_Y[j,j] ** 0.5)))
            else:
                s.append(0)
        
        lst.append(np.max(np.array(s)))

    return np.quantile(np.array(lst), ab)


In [11]:
def c_2_quantile_asymmetric(ab, rank_list, Sigma, delta_L, delta_U, n_iter, rand):
    
    ab_p = 1 - ab
    ab_p = ab_p / 2
    ab_p = 1 - ab_p

    lst_l = []
    lst_r = []
    lst = []
    
    k = len(delta_L)
    p = int(len(Sigma) / (k + 1))
    
    Sigma_Y = Sigma[: p, : p]

    for a in range(n_iter):
        draw = rand.multivariate_normal(np.zeros(p * (k + 1)), Sigma)
        
        I_list = []
        
        for i in range(k):
                                        
            X_temp = draw.copy()
            X_temp = X_temp[((i + 1) * p) : ((i + 2) * p)]
            rank = np.max(np.array(rank_list))
            
            I_list.append(make_I_list([j+1 for j in range(rank_list[i])], X_temp, delta_L[i], delta_U[i]))

        s = []
        
        Y_temp = draw.copy()
        Y_temp = Y_temp[:p]
        
        all_indices = []
                                        
        for i in range(k):
                                        
            I_l = I_list[i]
            I_indices = np.where(I_l == 1)[0]
            all_indices = list(set(all_indices) | set(I_indices))
                
        for j in all_indices:
            if Sigma_Y[j,j] > 0: s.append(Y_temp[j] / (Sigma_Y[j,j] ** 0.5))
            else: s.append(0)
        
        lst_r.append(np.max(np.array(s)))
        lst_l.append(np.min(np.array(s)))
        lst.append(np.max(abs(np.array(s))))

    return([np.quantile(-1 * np.array(lst_l), ab_p), np.quantile(np.array(lst_r), ab_p)])


In [12]:
# projection CI

def CI_proj(rank_list, x_vec_list, y_vec, a, Sigma_Y, n_iter, quantile_dict, rand):

    th_hat_lst = []
    
    for i in range(len(x_vec_list)):
        x_vec = x_vec_list[i]
        for j in range(rank_list[i]):
            th_hat_lst.append(np.argsort(x_vec)[len(x_vec)-(j+1)])
            
    c_a = c_quantile(a, Sigma_Y, n_iter, rand = rand)
        
    CI_list = []
        
    for i in range(len(th_hat_lst)):  

        CI_lower_bound = y_vec[th_hat_lst[i]] - c_a * Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5
        CI_upper_bound = y_vec[th_hat_lst[i]] + c_a * Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5
        
        CI_list.append([CI_lower_bound, CI_upper_bound])

    return CI_list


In [13]:
# locally simultaneous CI - Petrou-Zeniou and Shaikh

def locally_simultaneous1(rank_list, x_vec_lst, y_vec, a, b, Sigma_Y, n_iter, quantile_dict, rand):

    N = len(y_vec)
    k = len(x_vec_lst)
    
    b_1, b_2 = b / 2, b / 2
    
    Sigma_X = Sigma[: N * k, : N * k]
    Sigma_Y = Sigma[N * k:, N * k:]

    try:
        d = quantile_dict[str(b_2) + str(Sigma_X) + "d"]
    except:
        d = d_quantile(1-b_2, k, Sigma_X, n_iter, rand = rand)
        quantile_dict[str(b_2) + str(Sigma_X) + "d"] = d
        
    U = [np.zeros((N, N)) for i in range(k)]
    
    th_hat_lst = []
    
    for i in range(len(x_vec_lst)):
        
        x_vec = x_vec_lst[i]
        for j in range(rank_list[i]):
            th_hat_lst.append(np.argsort(x_vec)[len(x_vec)-(j+1)])
        l = np.tile(x_vec, (len(x_vec), 1))
        r = np.transpose(l)

        U[i] = r - l + d
    
    p = p_j(U, Sigma_X, k, n_iter, rand = rand)
    J = J_hat(p, 1 - b_1)
            
    try:
        c_a = quantile_dict[str(a) + str(Sigma_Y) + "c"]
    except:
        c_a = c_quantile_local(1-a+b, Sigma_Y, J, n_iter, rand = rand)
        quantile_dict[str(a) + str(Sigma_Y) + "c"] = c_a
        
    CI_list = []
        
    for i in range(len(x_vec_lst)):  

        CI_lower_bound = y_vec[th_hat_lst[i]] - c_a * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        CI_upper_bound = y_vec[th_hat_lst[i]] + c_a * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        
        CI_list.append([CI_lower_bound, CI_upper_bound])

    return CI_list



In [14]:
# locally simultaneous CI - Zrnic and Fithian

def locally_simultaneous2(rank_list, x_vec_lst, y_vec, a, b, Sigma_Y, n_iter, quantile_dict, rand):

    N = len(y_vec)
    k = len(x_vec_lst)
    
    Sigma_X = Sigma[: N * k, : N * k]
    Sigma_Y = Sigma[N * k:, N * k:]
    
    th_hat_lst = [] 
    d = d_quantile(1 - b, k, Sigma_X, n_iter, rand = rand)
    
    J = np.array([])
    
    for i in range(len(x_vec_lst)):
        x_vec = x_vec_lst[i]
        for j in range(rank_list[i]):
            th_hat_lst.append(np.argsort(x_vec)[len(x_vec)-(j+1)])
        top = x_vec[th_hat_lst[i]]
        J_i = np.where(x_vec >= top - 2 * d)
        J = np.union1d(J, J_i)
        
    try:
        c_a = quantile_dict[str(a) + str(Sigma_Y) + "c"]
    except:
        c_a = c_quantile_local(1-a+b, Sigma_Y, J, n_iter, rand = rand)
        quantile_dict[str(a) + str(Sigma_Y) + "c"] = c_a
        
    CI_list = []
        
    for i in range(len(x_vec_lst)):  

        CI_lower_bound = y_vec[th_hat_lst[i]] - c_a * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        CI_upper_bound = y_vec[th_hat_lst[i]] + c_a * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        
        CI_list.append([CI_lower_bound, CI_upper_bound])

    return CI_list



In [15]:
def two_step_CI(rank_list, x_vec_lst, y_vec, a, b, Sigma, n_iter, quantile_dict, rand):
    
    N = len(y_vec)
    k = len(x_vec_lst)
    
    Sigma_X = Sigma[: N * k, : N * k]
    Sigma_Y = Sigma[N * k:, N * k:]

    th_hat_lst = [] 
    
    for i in range(len(x_vec_lst)):
        x_vec = x_vec_lst[i]
        th_hat_lst.append(np.argsort(x_vec)[len(x_vec)-rank_list[i]])

    L = [np.zeros((N, N)) for i in range(k)]
    U = [np.zeros((N, N)) for i in range(k)]
    d = d_quantile(1 - b, k, Sigma_X, n_iter, rand = rand)
    
    for i in range(len(x_vec_lst)):
        
        x_vec = x_vec_lst[i]
        l = np.tile(x_vec, (len(x_vec), 1))
        r = np.transpose(l)

        L[i] = r - l - d
        U[i] = r - l + d

    bd = c_2_quantile(1-a+b, rank_list, Sigma, L, U, int(n_iter), rand = rand)
    
    CI_list = []
    
    for i in range(k):
        th_hat = th_hat_lst[i]
        CI_t = [y_vec[th_hat] - bd * (Sigma_Y[th_hat, th_hat] ** 0.5), y_vec[th_hat] + bd * (Sigma_Y[th_hat, th_hat] ** 0.5)]
        CI_list.append(CI_t)

    return CI_list



In [16]:
def two_step_CI_asymmetric(rank_list, x_vec_lst, y_vec, a, b, Sigma, n_iter, quantile_dict, rand):

    N = len(y_vec)
    k = len(x_vec_lst)
    
    Sigma_X = Sigma[N :, N :]
    Sigma_Y = Sigma[: N, : N]

    th_hat_lst = [] 
    
    for i in range(len(x_vec_lst)):
        x_vec = x_vec_lst[i]
        for j in range(rank_list[i]):
            th_hat_lst.append(np.argsort(x_vec)[len(x_vec)-(j+1)])

    L = [np.zeros((N, N)) for i in range(k)]
    U = [np.zeros((N, N)) for i in range(k)]
    d = d_quantile(1 - b, k, Sigma_X, n_iter, rand = rand)
    
    for i in range(len(x_vec_lst)):
        
        x_vec = x_vec_lst[i]
        l = np.tile(x_vec, (len(x_vec), 1))
        r = np.transpose(l)

        L[i] = r - l - d
        U[i] = r - l + d

    bd = c_2_quantile_asymmetric(1 - a + b, rank_list, Sigma, L, U, int(n_iter), rand = rand)
    bd_l = bd[0]
    bd_r = bd[1]
    
    CI_list = []
        
    for i in range(len(th_hat_lst)):  

        CI_lower_bound = y_vec[th_hat_lst[i]] - bd_r * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        CI_upper_bound = y_vec[th_hat_lst[i]] + bd_l * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        
        CI_list.append([CI_lower_bound, CI_upper_bound])

    return CI_list


In [17]:
def zoom_CI_out(rank_list, x_vec_lst, y_vec, a, Sigma, n_iter, quantile_dict, rand):

    N = len(y_vec)
    k = len(x_vec_lst)
    
    Sigma_X = Sigma[N :, N :]
    Sigma_Y = Sigma[: N, : N]

    th_hat_lst = [] 
    
    for i in range(len(x_vec_lst)):
        x_vec = x_vec_lst[i]
        for j in range(rank_list[i]):
            th_hat_lst.append(np.argsort(x_vec)[len(x_vec)-(j+1)])

    

    r_l_hat = r_l(y_vec, rank_list, x_vec_lst, Sigma_Y, a)
    r_u_hat = r_u(y_vec, rank_list, x_vec_lst, Sigma_Y, a)
    
    CI_list = []
        
    for i in range(len(th_hat_lst)):  

        CI_lower_bound = y_vec[th_hat_lst[i]] - r_l_hat
        CI_upper_bound = y_vec[th_hat_lst[i]] + r_u_hat
        
        CI_list.append([CI_lower_bound, CI_upper_bound])

    return CI_list


In [18]:
def individual_run(param_list, q_dict = {}):
    
    print("Start")

    if len(param_list) == 8:

        rank_list, mu_vec_x, mu_vec_y, Sigma, n_test, s_level, scale_factor, seed = param_list[0], param_list[1], param_list[2], param_list[3], param_list[4], param_list[5], param_list[6], param_list[7]

    if len(param_list) == 7:

        rank_list, mu_vec_x, mu_vec_y, Sigma, n_test, s_level, seed = param_list[0], param_list[1], param_list[2], param_list[3], param_list[4], param_list[5], param_list[6]
        
    mu_vec = np.concatenate((mu_vec_x, mu_vec_y))
    
    p = len(mu_vec_y)
    Sigma_Y = Sigma[p: , p:]
    
    top_k = max(rank_list)
    
    rng = np.random.default_rng(seed)
    
    p_cov_prob = [0 for i in range(5)]
    ts_cov_prob = [0 for i in range(5)]
    LS1_cov_prob = [0 for i in range(5)]
    LS2_cov_prob = [0 for i in range(5)]
    zoom_cov_prob = [0 for i in range(5)]

    p_length = [0 for i in range(5)]
    ts_length = [0 for i in range(5)]
    LS1_length = [0 for i in range(5)]
    LS2_length = [0 for i in range(5)]
    zoom_length = [0 for i in range(5)]
    
    draw = rng.multivariate_normal(np.zeros(len(mu_vec)), Sigma)
  
    for n in range(5):
        
        a_vec = draw / (scale_factor ** (n * 0.5)) + mu_vec
        y_vec = a_vec[N * k :]
        
        x_vec_lst = []
        for i in range(N):
            x_vec_lst.append(a_vec[i * k : (i + 1) * k])
        
        ts_CI = two_step_CI(rank_list, x_vec_lst, y_vec, s_level, s_level / 10, np.array(Sigma) / (scale_factor ** n), n_test, q_dict, rand = rng)
        p_CI = CI_proj(rank_list, x_vec_lst, y_vec, 1-s_level, np.array(Sigma_Y) / (scale_factor ** n), n_test, q_dict, rand = rng)
        LS1_CI = locally_simultaneous1(rank_list, x_vec_lst, y_vec, s_level, s_level / 10, np.array(Sigma_Y) / (scale_factor ** n), n_test, q_dict, rand = rng)
        LS2_CI = locally_simultaneous2(rank_list, x_vec_lst, y_vec, s_level, s_level / 10, np.array(Sigma_Y) / (scale_factor ** n), n_test, q_dict, rand = rng)
        zoom_CI = zoom_CI_out(rank_list, x_vec_lst, y_vec, s_level, np.array(Sigma_Y) / (scale_factor ** n), n_test, q_dict, rand = rng)
            
        p_add = 1
        ts_add = 1
        LS1_add = 1
        LS2_add = 1
        zoom_add = 1
        
        p_vol = 1
        ts_vol = 1
        LS1_vol = 1
        LS2_vol = 1
        zoom_vol = 1
        
        for i in range(len(rank_list)):
            
            mu_vec_i = mu_vec[i * k : (i + 1) * k]
            th_hat = np.argsort(x_vec_lst[i])[len(y_vec)-rank_list[i]]
        
            if (p_CI[i][0] <= mu_vec_y[th_hat] <= p_CI[i][1]): p_add *= 1
            else: p_add *= 0
                
            if (ts_CI[i][0] <= mu_vec_y[th_hat] <= ts_CI[i][1]): ts_add *= 1
            else: ts_add *= 0
                
            if (LS1_CI[i][0] <= mu_vec_y[th_hat] <= LS1_CI[i][1]): LS1_add *= 1
            else: LS1_add *= 0
                
            if (LS2_CI[i][0] <= mu_vec_y[th_hat] <= LS2_CI[i][1]): LS2_add *= 1
            else: LS2_add *= 0
                
            if (zoom_CI[i][0] <= mu_vec_y[th_hat] <= zoom_CI[i][1]): zoom_add *= 1
            else: zoom_add *= 0

            p_vol *= p_CI[i][1] - p_CI[i][0]
            ts_vol *= ts_CI[i][1] - ts_CI[i][0]
            LS1_vol *= LS1_CI[i][1] - LS1_CI[i][0]
            LS2_vol *= LS2_CI[i][1] - LS2_CI[i][0]
            zoom_vol *= zoom_CI[i][1] - zoom_CI[i][0]

        p_cov_prob[n] += p_add
        ts_cov_prob[n] += ts_add
        LS1_cov_prob[n] += LS1_add
        LS2_cov_prob[n] += LS2_add
        zoom_cov_prob[n] += zoom_add

        p_length[n] += p_vol
        ts_length[n] += ts_vol
        LS1_length[n] += LS1_vol
        LS2_length[n] += LS2_vol
        zoom_length[n] += zoom_vol
        
    print("Done")
        
    return(p_cov_prob,
           ts_cov_prob,
           LS1_cov_prob,
           LS2_cov_prob,
           zoom_cov_prob,
           p_length,
           ts_length,
           LS1_length,
           LS2_length,
           zoom_length)


In [19]:
X_list = [np.array([np.tan(i - 3) for i in range(1,6)]),
          np.concatenate((np.array([np.tan(i - 3) for i in range(1,6)]), -1 * np.array([np.tan(i - 3) for i in range(1,6)]))),
          np.array([np.tan(i - 5.5) for i in range(1,11)]),
          np.concatenate((np.array([np.tan(i - 5.5) for i in range(1,11)]), -1 * np.array([np.tan(i - 5.5) for i in range(1,11)]))),
          np.concatenate((np.array([1,0,0,0,0]),np.array([0,1,0,0,0]))),
          np.concatenate((np.array([5,0,0,0,0]),np.array([0,5,0,0,0]))),
          np.concatenate((np.array([1,1,0,0,0]),np.array([0,0,1,1,0]))),
          np.concatenate((np.array([5,5,0,0,0]),np.array([0,0,5,5,0]))),
          np.array([1,1,0,0,0]),
          np.array([5,5,0,0,0]),
          np.concatenate((np.zeros(5),np.zeros(5))),
          np.concatenate((np.array([0.1,0,0,0,0]),np.array([0,0.1,0,0,0]))),
          np.concatenate((np.array([0.01,0,0,0,0]),np.array([0,0.01,0,0,0]))),
          np.concatenate((np.array([0.1,0.1,0,0,0]),np.array([0,0,0.1,0.1,0]))),
          np.concatenate((np.array([0.01,0.01,0,0,0]),np.array([0,0,0.01,0.01,0]))),
          np.array([0.1,0.1,0,0,0]),
          np.array([0.01,0.01,0,0,0]),
          np.concatenate((np.array([1,0,0,0,0,0,0,0,0,0]),np.array([0,1,0,0,0,0,0,0,0,0]))),
          np.concatenate((np.array([5,0,0,0,0,0,0,0,0,0]),np.array([0,5,0,0,0,0,0,0,0,0]))),
          np.concatenate((np.array([1,1,0,0,0,0,0,0,0,0]),np.array([0,0,1,1,0,0,0,0,0,0]))),
          np.concatenate((np.array([5,5,0,0,0,0,0,0,0,0]),np.array([0,0,5,5,0,0,0,0,0,0]))),
          np.concatenate((np.array([0.1,0,0,0,0,0,0,0,0,0]),np.array([0,0.1,0,0,0,0,0,0,0,0]))),
          np.concatenate((np.array([0.01,0,0,0,0,0,0,0,0,0]),np.array([0,0.01,0,0,0,0,0,0,0,0]))),
          np.concatenate((np.array([0.1,0.1,0,0,0,0,0,0,0,0]),np.array([0,0,0.1,0.1,0,0,0,0,0,0]))),
          np.concatenate((np.array([0.01,0.01,0,0,0,0,0,0,0,0]),np.array([0,0,0.01,0.01,0,0,0,0,0,0]))),
          np.array([1,1,0,0,0,0,0,0,0,0]),
          np.array([5,5,0,0,0,0,0,0,0,0]),
          np.array([0.1,0.1,0,0,0,0,0,0,0,0]),
          np.array([0.01,0.01,0,0,0,0,0,0,0,0])] #,
#           np.concatenate((np.array([1,1,0,0,0,0,0,0,0,0]),np.array([1,1,0,0,0,0,0,0,0,0]))),
#           np.concatenate((np.array([5,5,0,0,0,0,0,0,0,0]),np.array([5,5,0,0,0,0,0,0,0,0]))),
#           np.concatenate((np.array([0.1,0.1,0,0,0,0,0,0,0,0]),np.array([0.1,0.1,0,0,0,0,0,0,0,0]))),
#           np.concatenate((np.array([0.01,0.01,0,0,0,0,0,0,0,0]),np.array([0.01,0.01,0,0,0,0,0,0,0,0]))),
#           np.concatenate((np.array([1,1,0,0,0]),np.array([1,1,0,0,0]))),
#           np.concatenate((np.array([5,5,0,0,0]),np.array([5,5,0,0,0]))),
#           np.concatenate((np.array([0.1,0.1,0,0,0]),np.array([0.1,0.1,0,0,0]))),
#           np.concatenate((np.array([0.01,0.01,0,0,0]),np.array([0.01,0.01,0,0,0])))]


Y_list = [np.zeros(5),
          np.zeros(5),
          np.zeros(10),
          np.zeros(10),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(5),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10),
          np.zeros(10)] # ,
#           np.zeros(10),
#           np.zeros(10),
#           np.zeros(10),
#           np.zeros(10),
#           np.zeros(5),
#           np.zeros(5),
#           np.zeros(5),
#           np.zeros(5)]

Sigma_list = [np.identity(10),
              np.identity(15),
              np.identity(20),
              np.identity(30),
              np.identity(15),
              np.identity(15),
              np.identity(15),
              np.identity(15),
              np.identity(10),
              np.identity(10),
              np.identity(15),
              np.identity(15),
              np.identity(15),
              np.identity(15),
              np.identity(15),
              np.identity(10),
              np.identity(10),
              np.identity(30),
              np.identity(30),
              np.identity(30),
              np.identity(30),
              np.identity(30),
              np.identity(30),
              np.identity(30),
              np.identity(30),
              np.identity(20),
              np.identity(20),
              np.identity(20),
              np.identity(20)] #,
#               np.identity(30),
#               np.identity(30),
#               np.identity(30),
#               np.identity(30),
#               np.identity(15),
#               np.identity(15),
#               np.identity(15),
#               np.identity(15)]

rank_list_list = [[1],
                  [1, 1],
                  [1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1],
                  [1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1],
                  [1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1, 1],
                  [1],
                  [1],
                  [1],
                  [1]] #,
#                   [1, 2],
#                   [1, 2],
#                   [1, 2],
#                   [1, 2],
#                   [1, 2],
#                   [1, 2],
#                   [1, 2],
#                   [1, 2]]

df_tot = pd.DataFrame({"p_cov_prob":[], "ts_cov_prob":[], "p_length":[], "ts_length":[], "Design":[], "Corr":[]})


In [20]:
rng = np.random.default_rng(90)

for design in range(len(X_list)):
    
    X = X_list[design]
    Y = Y_list[design]
    Sigma = Sigma_list[design]
    rank_list = rank_list_list[design]
    
    n_test = 1000
    s_level = 0.05
    
    for corr in ["Low", "Med", "High"]:
        
        if corr == "Low": Sigma[Sigma != 1] = 0
        elif corr == "Med": Sigma[Sigma != 1] = 0.5
        elif corr == "High": Sigma[Sigma != 1] = 0.95
            
        seed_seq = np.random.SeedSequence()
        seeds = seed_seq.spawn(n_test)

        pool = multiprocessing.Pool(55)
        p_cov_prob, shaikh_cov_prob, LS1_cov_prob, LS2_cov_prob, zoom_cov_prob, p_length, shaikh_length, LS1_length, LS2_length, zoom_length = zip(*pool.map(individual_run, [[rank_list, X, Y, Sigma, int(n_test), s_level, 10, seeds[i]] for i in range(n_test)]))
        
        df_temp = pd.DataFrame({"p_cov_prob":[np.sum(np.matrix(p_cov_prob), axis=0)], 
                                "ts_cov_prob":[np.sum(np.matrix(shaikh_cov_prob), axis=0)], 
                                "LS1_cov_prob":[np.sum(np.matrix(LS1_cov_prob), axis=0)], 
                                "LS2_cov_prob":[np.sum(np.matrix(LS2_cov_prob), axis=0)], 
                                "zoom_cov_prob":[np.sum(np.matrix(zoom_cov_prob), axis=0)], 
                                "p_length":[np.sum(np.matrix(p_length), axis=0)], 
                                "ts_length":[np.sum(np.matrix(shaikh_length), axis=0)],
                                "LS1_length":[np.sum(np.matrix(LS1_length), axis=0)], 
                                "LS2_length":[np.sum(np.matrix(LS2_length), axis=0)], 
                                "zoom_length":[np.sum(np.matrix(zoom_length), axis=0)]})
        df_temp["Design"] = design + 1
        df_temp["Corr"] = corr
        
        df_tot = pd.concat([df_tot, df_temp])
        
  

Start
8
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]
[0, 3, 1, 2, 4]
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]
2.5758293035489004
6.478291563284641
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]
[0, 3, 1, 2, 4]
0.8145487463019763
4.745932315842982
0.7899203586417092
4.607597416417738
0.7572548387227913
2.6616482368606653
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]
[0, 3, 1, 2, 4]
0.25758293035489005
4.488934751195328
0.24977054744123742
4.016031985453243
0.23939797998185106
2.3357566645096957
0.22414027276265103
0.6665658667986742
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]
[0, 3, 1, 2, 4]
0.08145487463019763
4.407664985455032
0.07898438223414514
3.828962570766553
0.07570428839860444
2.2327007006214337
0.07087937772945595
0.6399440644954979
0.06197950376932

KeyboardInterrupt: 

In [ ]:
df_tot

In [ ]:
df_tot.to_csv("Final_Simulation_Study.csv")